In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mp.binds.copter import *
from mp.binds.math import *
from mp.sim.rigidbody import *
from mp.utils.numeric import *

In [ ]:
copter_mass = 1
copter_inertia = np.diag([0.01, 0.01, 0.02])
motor_delay = 0.1

In [ ]:
def simulate(controller, input_v, end, dt = 0.01, motor_delay = 0):
    rb_props = RigidbodyProperties(copter_mass, copter_inertia)
    rb = Rigidbody(rb_props)
    
    states = [state_s()]
    t = 0
    
    motor_coeff = dt / (dt + motor_delay)
    force = np.zeros(3)
    torque = np.zeros(3)
    while t < end:
        state = states[-1]
        actuation = controller.update(input_v, state, dt)
        
        force_v = actuation.thrust * rot_v(state.rotationq.as_vector(), np.array([0, 0, -1]))
        force = motor_coeff * force_v + (1 - motor_coeff) * force
        torque = motor_coeff * actuation.torque.get_base() + (1 - motor_coeff) * torque
        states.append(rb.update(dt, force, torque))
        
        t += dt
    return np.arange(0, end, dt), states[:-1]

In [ ]:
pid_params = copter_controller_pid_params_s(20, 8, 1.4, 0, 0, 0)
copter_params = copter_params_s(copter_mass, matrix3f(copter_inertia), 0)
pid = copter_controller_pid(pid_params, copter_params)

input_v = angular_controls_s(vector3f(2, -0.6, 1), 0)
t, res = simulate(pid, input_v, 1, motor_delay=motor_delay)

In [ ]:
acc = np.array([s.acceleration.get_base() for s in res])
vel = np.array([s.velocity.get_base() for s in res])
rot = np.array([s.rotationq.as_vector() for s in res])
ang = np.array([s.angular_velocity.get_base() for s in res])
drift = np.array([s.gyroscope_drift.get_base() for s in res])

fig, ax = plt.subplots(2, 2, figsize=[12, 9])

ax[0,0].plot(t, acc[:, 0], label="x")
ax[0,0].plot(t, acc[:, 1], label="y")
ax[0,0].plot(t, acc[:, 2], label="z")
ax[0,0].legend()

ax[0,1].plot(t, rot[:, 1], label="x")
ax[0,1].plot(t, rot[:, 2], label="y")
ax[0,1].plot(t, rot[:, 3], label="z")
ax[0,1].legend()

ax[1,0].plot(t, ang[:, 0], label="x")
ax[1,0].plot(t, ang[:, 1], label="y")
ax[1,0].plot(t, ang[:, 2], label="z")
ax[1,0].legend()

ax[1,1].plot(t, drift[:, 0], label="x")
ax[1,1].plot(t, drift[:, 1], label="y")
ax[1,1].plot(t, drift[:, 2], label="z")
ax[1,1].legend()